In [1]:
import polars as pl
from sklearn.ensemble import IsolationForest
import structlog
import config

logger = structlog.get_logger()

In [2]:
# 1. Use Polars for Lightning-Fast I/O & Memory Safety
lf = pl.scan_parquet(config.GOLD_DATA_PATH)
lf = lf.drop_nulls(subset=['sales_last_year'])

In [3]:
# 2. Engineer Contextual Ratios (Defeating the "Volume Outlier" Trap)
# If we feed absolute dollars, the algorithm blindly flags high-volume departments (like Grocery).
# By converting sales into dimensionless Ratios, we mathematically force the tree to hunt for Decoupling Deviations!
# (Adding +1 to denominator prevents divide-by-zero crashes on zero-sales closed weeks).
lf = lf.with_columns([
    (pl.col('weekly_sales') / (pl.col('sales_last_year') + 1)).alias('yoy_growth_ratio'),
    (pl.col('weekly_sales') / (pl.col('rolling_4_wk_sales_avg') + 1)).alias('trend_deviation_ratio'),
    (pl.col('total_markdown') / (pl.col('weekly_sales') + 1)).alias('markdown_intensity_ratio')
])

# Engineer the Advanced Decoupling Ratios
lf = lf.with_columns([
    (pl.col('markdown_intensity_ratio') / (pl.col('yoy_growth_ratio') + 0.01)).alias('markdown_cannibalization'),
    (pl.col('weekly_sales') / (pl.col('store_size'))).alias('footprint_yield'),
    pl.when(pl.col('isholiday') == 1)
        .then(pl.col('trend_deviation_ratio'))
        .otherwise(pl.lit(1.0))
        .alias('holiday_miss_severity')
])


In [4]:
 
# Contextual Isolation
# We feed it the completely dimensionless RATIOS!
features = [
    'yoy_growth_ratio', 
    'trend_deviation_ratio', 
    'markdown_intensity_ratio',
    'markdown_cannibalization',
    'footprint_yield',
    'holiday_miss_severity'
]

logger.info("Extracting Contextual Feature Matrix (Ratios)...", features=features)

2026-04-25 11:52:42 [info     ] Extracting Contextual Feature Matrix (Ratios)... features=['yoy_growth_ratio', 'trend_deviation_ratio', 'markdown_intensity_ratio', 'markdown_cannibalization', 'footprint_yield', 'holiday_miss_severity']


In [5]:

    
# Isolate just the features we need for math and fill nulls efficiently in Rust
# Then cast to Pandas natively because Scikit-Learn mandates Numpy arrays!
X_train = lf.select(features).fill_null(0).collect().to_pandas()

logger.info("Training Isolation Forest to hunt dimensional decoupling...", rows=len(X_train))
# contamination=0.005 means we mandate the algorithm isolates the most extreme 0.5% of the database
iso_forest = IsolationForest(contamination=0.005, random_state=42, n_jobs=-1)

# Train and Score
anomaly_predictions = iso_forest.fit_predict(X_train)


2026-04-25 11:52:42 [info     ] Training Isolation Forest to hunt dimensional decoupling... rows=303121


In [6]:

# 3. Re-attach the predictions back to the core Polars DataFrame
df_results = lf.collect().with_columns(
    pl.Series("Anomaly_Flag", anomaly_predictions)
)

# Isolation forest outputs -1 for anomalous, 1 for normal
# We use Polars fast filtering syntax
anomalies = df_results.filter(pl.col('Anomaly_Flag') == -1)


In [7]:

# Isolate relevant business attributes to show exactly WHY they were flagged
business_view = anomalies.select([
    'store', 'dept', 'date', 'isholiday', 'weekly_sales', 
    'sales_last_year', 'rolling_4_wk_sales_avg', 
    'yoy_growth_ratio', 'trend_deviation_ratio', 'total_markdown',
    'markdown_intensity_ratio', 'markdown_cannibalization', 
    'footprint_yield', 'holiday_miss_severity'
])

In [31]:
business_view.sample(10)

store,dept,date,isholiday,weekly_sales,sales_last_year,rolling_4_wk_sales_avg,yoy_growth_ratio,trend_deviation_ratio,total_markdown,markdown_intensity_ratio,markdown_cannibalization,footprint_yield,holiday_miss_severity
i32,i32,datetime[μs],i32,f64,f64,f64,f64,f64,f32,f64,f64,f64,f64
26,78,2011-11-25 00:00:00,1,0.0,0.0,0.0,0.0,0.0,46031.832031,46031.832031,4.6032e6,0.0,0.0
14,99,2012-02-03 00:00:00,0,345.0,0.0,0.0,345.0,345.0,143777.921875,415.543127,1.204438,0.001717,1.0
11,45,2012-02-03 00:00:00,0,0.0,14.47,15.83,0.0,0.0,96513.929688,96513.929688,9.6514e6,0.0,1.0
40,78,2012-02-10 00:00:00,1,0.0,0.0,0.0,0.0,0.0,33601.832031,33601.832031,3.3602e6,0.0,0.0
12,99,2012-02-17 00:00:00,0,0.0,0.0,0.0,0.0,0.0,46456.714844,46456.714844,4.6457e6,0.0,1.0
13,43,2011-11-11 00:00:00,0,0.0,0.0,0.0,0.0,0.0,75070.484375,75070.484375,7.5070e6,0.0,1.0
30,56,2012-02-10 00:00:00,1,254.72,0.0,15.0,254.72,15.92,4079.009766,15.951078,0.06262,0.005925,15.92
12,99,2012-01-06 00:00:00,0,0.0,0.0,0.0,0.0,0.0,55824.710938,55824.710938,5.5825e6,0.0,1.0
41,45,2011-11-25 00:00:00,1,0.0,0.0,21.4025,0.0,0.0,83589.453125,83589.453125,8.3589e6,0.0,0.0
